# Mei's Bulkseq Analysis

##### Tommy Tang

In [2]:
#Libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.ticker import MaxNLocator, MultipleLocator
from PIL import Image
import re
import io
from typing import Union
from scipy.stats import chi2_contingency, chi2
import itertools

## Task 1: Proprocessing and Exploration 

In [ ]:
#Read the Excel file and get TPM columns
#df = pd.read_excel('/Users/tommy/OneDrive/Documents/My_Documents/Work/LTRI/Zhen Lab/Projects/TRAPseq and Bulkseq/Data/Mei_RNASeq_Analysis_March272026/Expression Browser.xls')
df = pd.read_excel('/mnt/c/Users/tommy/OneDrive/Documents/My_Documents/Work/LTRI/Zhen Lab/Projects/TRAPseq and Bulkseq/Data/Mei_RNASeq_Analysis_April212026/Expression Browser 12 samples.xlsx', sheet_name=0)
df_TPM = pd.concat([df.Name, df.filter(like="TPM")], axis=1)

,Name,A11-18_S78 TPM,A12-15_S80 TPM,G11-18_S79 TPM,G12-15_S81 TPM,A02-04_S6 TPM,A29-19_S7 TPM,G02-04_S8 TPM,G29-19_S9 TPM,G01-23_S11 TPM,A01-23_S10 TPM,G11-26_S13 TPM,A11-26_S12 TPM
0,Y74C9A.6,0.777513,0.000000,1.262344,0.000000,2.000737,1.725581,1.127245,0.000000,0.000000,0.411211,0.316059,0.358549
1,homt-1,11.650486,7.945634,8.842539,10.256303,7.220710,10.090395,12.565246,12.485014,12.346182,9.167428,15.131842,12.055597
2,nlp-40,93.098424,196.855230,1012.768004,2420.186732,97.059571,182.129394,491.391288,1169.400214,2013.030829,200.988388,932.222648,128.428231
3,rcor-1,12.106878,13.301003,8.940562,4.980565,9.361567,11.206670,5.663545,7.420557,6.193710,9.904287,6.957902,12.403761
4,gene:WBGene00235381,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
46921,gene:WBGene00014471,1.173468,4.743980,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.487081,3.723746,0.954029,0.000000
46922,gene:WBGene00014472,3399.268422,6707.010761,2635.425420,2491.475841,2964.627507,5260.217293,1179.165386,2193.867682,1088.949610,4228.782541,3518.076234,4779.715690
46923,gene:WBGene00010966,1589.085379,2604.444920,1145.776057,765.766461,948.514659,1699.059115,322.913815,622.710445,319.367849,2110.964799,1942.028300,1824.270228
46924,gene:WBGene00010967,4926.265146,4705.392251,3737.706698,1151.789208,4425.967981,4479.810661,1409.880921,1840.385693,547.906147,3893.733911,1010.193716,7601.784764


In [28]:
df_TPM

,Gene Name,L3_A11-18_S78 TPM,L2D_A12-15_S80 TPM,L3_G11-18_S79 TPM,L2D_G12-15_S81 TPM,L3_A02-04_S6 TPM,L2D_A29-19_S7 TPM,L3_G02-04_S8 TPM,L2D_G29-19_S9 TPM,L2D_G01-23_S11 TPM,L2D_A01-23_S10 TPM,L3_G11-26_S13 TPM,L3_A11-26_S12 TPM
0,Y74C9A.6,0.777513,0.000000,1.262344,0.000000,2.000737,1.725581,1.127245,0.000000,0.000000,0.411211,0.316059,0.358549
1,homt-1,11.650486,7.945634,8.842539,10.256303,7.220710,10.090395,12.565246,12.485014,12.346182,9.167428,15.131842,12.055597
2,nlp-40,93.098424,196.855230,1012.768004,2420.186732,97.059571,182.129394,491.391288,1169.400214,2013.030829,200.988388,932.222648,128.428231
3,rcor-1,12.106878,13.301003,8.940562,4.980565,9.361567,11.206670,5.663545,7.420557,6.193710,9.904287,6.957902,12.403761
4,gene:WBGene00235381,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
46921,gene:WBGene00014471,1.173468,4.743980,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.487081,3.723746,0.954029,0.000000
46922,gene:WBGene00014472,3399.268422,6707.010761,2635.425420,2491.475841,2964.627507,5260.217293,1179.165386,2193.867682,1088.949610,4228.782541,3518.076234,4779.715690
46923,gene:WBGene00010966,1589.085379,2604.444920,1145.776057,765.766461,948.514659,1699.059115,322.913815,622.710445,319.367849,2110.964799,1942.028300,1824.270228
46924,gene:WBGene00010967,4926.265146,4705.392251,3737.706698,1151.789208,4425.967981,4479.810661,1409.880921,1840.385693,547.906147,3893.733911,1010.193716,7601.784764


In [30]:
#Rename dataframes
df_TPM = df_TPM.rename(columns={"Name": "Gene Name",
                                "A11-18_S78 TPM": "L3_A11-18_S78 TPM",
                                "A12-15_S80 TPM": "L2D_A12-15_S80 TPM",
                                "G11-18_S79 TPM": "L3_G11-18_S79 TPM",
                                "G12-15_S81 TPM": "L2D_G12-15_S81 TPM",
                                "A02-04_S6 TPM": "L3_A02-04_S6 TPM",
                                "A29-19_S7 TPM": "L2D_A29-19_S7 TPM",
                                "G02-04_S8 TPM": "L3_G02-04_S8 TPM",
                                "G29-19_S9 TPM": "L2D_G29-19_S9 TPM",
                                "G01-23_S11 TPM": "L2D_G01-23_S11 TPM",
                                "A01-23_S10 TPM": "L2D_A01-23_S10 TPM",
                                "G11-26_S13 TPM": "L3_G11-26_S13 TPM",
                                "A11-26_S12 TPM": "L3_A11-26_S12 TPM"
    
})
df_TPM_GA_ratio = pd.DataFrame({"Name": df_TPM["Gene Name"],
                                "L3_11-18_S78 TPM": df_TPM["L3_G11-18_S79 TPM"] / df_TPM["L3_A11-18_S78 TPM"],
                                "L2D_12-15_S80 TPM": df_TPM["L2D_G12-15_S81 TPM"] / df_TPM["L2D_A12-15_S80 TPM"],
                                "L3_02-04_S6 TPM": df_TPM["L3_G02-04_S8 TPM"] / df_TPM["L3_A02-04_S6 TPM"], 
                                "L2D_29-19_S7 TPM": df_TPM["L2D_G29-19_S9 TPM"] / df_TPM["L2D_A29-19_S7 TPM"],
                                "L2D_01-23_S11 TPM": df_TPM["L2D_G01-23_S11 TPM"] / df_TPM["L2D_A01-23_S10 TPM"],
                                "L3_11-26_S13 TPM": df_TPM["L3_G11-26_S13 TPM"] / df_TPM["L3_A11-26_S12 TPM"],
                                "Log2FC L3_11-18_S78 TPM": np.log2(df_TPM["L3_G11-18_S79 TPM"] / df_TPM["L3_A11-18_S78 TPM"]),
                                "Log2FC L2D_12-15_S80 TPM": np.log2(df_TPM["L2D_G12-15_S81 TPM"] / df_TPM["L2D_A12-15_S80 TPM"]),
                                "Log2FC L3_02-04_S6 TPM": np.log2(df_TPM["L3_G02-04_S8 TPM"] / df_TPM["L3_A02-04_S6 TPM"]), 
                                "Log2FC L2D_29-19_S7 TPM": np.log2(df_TPM["L2D_G29-19_S9 TPM"] / df_TPM["L2D_A29-19_S7 TPM"]),
                                "Log2FC L2D_01-23_S11 TPM": np.log2(df_TPM["L2D_G01-23_S11 TPM"] / df_TPM["L2D_A01-23_S10 TPM"]),
                                "Log2FC L3_11-26_S13 TPM": np.log2(df_TPM["L3_G11-26_S13 TPM"] / df_TPM["L3_A11-26_S12 TPM"])        
            
            })
df_TPM_GA_ratio

/home/tommytang111/miniconda/envs/ml/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Name,L3_11-18_S78 TPM,L2D_12-15_S80 TPM,L3_02-04_S6 TPM,L2D_29-19_S7 TPM,L2D_01-23_S11 TPM,L3_11-26_S13 TPM,Log2FC L3_11-18_S78 TPM,Log2FC L2D_12-15_S80 TPM,Log2FC L3_02-04_S6 TPM,Log2FC L2D_29-19_S7 TPM,Log2FC L2D_01-23_S11 TPM,Log2FC L3_11-26_S13 TPM
0,Y74C9A.6,1.623567,NaN,0.563415,0.000000,0.000000,0.881494,0.699167,NaN,-0.827730,-inf,-inf,-0.181977
1,homt-1,0.758985,1.290810,1.740168,1.237317,1.346744,1.255171,-0.397858,0.368276,0.799226,0.307215,0.429476,0.327884
2,nlp-40,10.878466,12.294247,5.062780,6.420711,10.015657,7.258705,3.443403,3.619911,2.339930,2.682733,3.324185,2.859712
3,rcor-1,0.738470,0.374450,0.604978,0.662155,0.625356,0.560951,-0.437389,-1.417154,-0.725045,-0.594758,-0.677249,-0.834053
4,gene:WBGene00235381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
46921,gene:WBGene00014471,0.000000,0.000000,NaN,NaN,0.667898,inf,-inf,-inf,NaN,NaN,-0.582301,inf
46922,gene:WBGene00014472,0.775292,0.371473,0.397745,0.417068,0.257509,0.736043,-0.367188,-1.428669,-1.330085,-1.261646,-1.957305,-0.442138
46923,gene:WBGene00010966,0.721029,0.294023,0.340442,0.366503,0.151290,1.064551,-0.471872,-1.766000,-1.554521,-1.448103,-2.724612,0.090245
46924,gene:WBGene00010967,0.758730,0.244781,0.318547,0.410818,0.140715,0.132889,-0.398341,-2.030438,-1.650420,-1.283430,-2.829154,-2.911706


In [31]:
#Set index to gene names
df_TPM_GA_ratio.set_index('Name', inplace=True)

In [32]:
#Data exploration
df_TPM_GA_ratio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 46926 entries, Y74C9A.6 to gene:WBGene00014473
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   L3_11-18_S78 TPM          21792 non-null  float64
 1   L2D_12-15_S80 TPM         23291 non-null  float64
 2   L3_02-04_S6 TPM           21068 non-null  float64
 3   L2D_29-19_S7 TPM          23282 non-null  float64
 4   L2D_01-23_S11 TPM         22617 non-null  float64
 5   L3_11-26_S13 TPM          23993 non-null  float64
 6   Log2FC L3_11-18_S78 TPM   21792 non-null  float64
 7   Log2FC L2D_12-15_S80 TPM  23291 non-null  float64
 8   Log2FC L3_02-04_S6 TPM    21068 non-null  float64
 9   Log2FC L2D_29-19_S7 TPM   23282 non-null  float64
 10  Log2FC L2D_01-23_S11 TPM  22617 non-null  float64
 11  Log2FC L3_11-26_S13 TPM   23993 non-null  float64
dtypes: float64(12)
memory usage: 4.7+ MB


In [33]:
df_TPM_GA_ratio.isna().sum()

L3_11-18_S78 TPM            25134
L2D_12-15_S80 TPM           23635
L3_02-04_S6 TPM             25858
L2D_29-19_S7 TPM            23644
L2D_01-23_S11 TPM           24309
L3_11-26_S13 TPM            22933
Log2FC L3_11-18_S78 TPM     25134
Log2FC L2D_12-15_S80 TPM    23635
Log2FC L3_02-04_S6 TPM      25858
Log2FC L2D_29-19_S7 TPM     23644
Log2FC L2D_01-23_S11 TPM    24309
Log2FC L3_11-26_S13 TPM     22933
dtype: int64

In [ ]:
#Drop NaN values
df_TPM_GA_ratio = df_TPM_GA_ratio.dropna()
df_TPM_GA_ratio.isna().sum()

L3_11-18_S78 TPM            0
L2D_12-15_S80 TPM           0
L3_02-04_S6 TPM             0
L2D_29-19_S7 TPM            0
L2D_01-23_S11 TPM           0
L3_11-26_S13 TPM            0
Log2FC L3_11-18_S78 TPM     0
Log2FC L2D_12-15_S80 TPM    0
Log2FC L3_02-04_S6 TPM      0
Log2FC L2D_29-19_S7 TPM     0
Log2FC L2D_01-23_S11 TPM    0
Log2FC L3_11-26_S13 TPM     0
dtype: int64

In [38]:
#Write out to excel
df_TPM_GA_ratio.to_excel('../Data/Mei_RNASeq_Analysis_April212026/df_TPM_GA_ratio.xlsx')

In [ ]:
#Filter out zeros
df_zero_filtered = df[(df>10).all(axis=1)]
df_zero_filtered

,A11-18 TPM,A12-15 TPM,G11-18 TPM,G12-15 TPM
Name,,,,
aagr-2,70.089847,62.585826,25.076285,70.953900
aagr-3,92.666468,76.388061,67.600681,47.534703
aagr-4,53.495709,53.926668,19.520043,11.133615
aak-1,13.905526,21.575494,56.266428,14.317476
aak-2,24.816848,32.222982,46.218244,24.527196
...,...,...,...,...
ztf-7,121.831763,139.072830,128.571752,121.061646
ztf-8,21.034776,21.091424,16.190053,12.167251
zyg-11,22.417813,30.730695,22.257547,21.245161


In [ ]:
#Find enriched genes in L2D and L3
df_zero_filtered = df_zero_filtered.sort_values(by=['G11-18 TPM', 'G12-15 TPM'], ascending=False)
df_rih_enriched = df_zero_filtered[(df_zero_filtered['G11-18 TPM']> 10 * df_zero_filtered['A11-18 TPM']) & (df_zero_filtered['G12-15 TPM'] > 10 * df_zero_filtered['A12-15 TPM'])]
df_rih_enriched.head(13)

,A11-18 TPM,A12-15 TPM,G11-18 TPM,G12-15 TPM
Name,,,,
nlp-71,121.000891,270.418360,14762.621939,24388.648962
rpl-22,913.453589,1290.676273,13059.914820,18133.134176
R11D1.12,47.662478,111.857693,2597.827725,1517.299024
C16B8.3,211.559367,151.176860,2386.301016,1958.116294
mihp-2,14.741627,53.070485,1333.114290,769.805230
nlp-40,114.221311,230.632518,1177.479615,2708.568150
Y37H2A.14,71.588808,73.198991,1118.376421,1274.912056
F15D4.5,53.859165,71.537136,604.790617,892.171024
F13B6.3,15.250771,13.649004,503.856854,228.790982


In [45]:
#Find enriched genes in L2D or L3 only.
df_zero_filtered = df_zero_filtered.sort_values(by=['G11-18 TPM', 'G12-15 TPM'], ascending=False)
df_rih_enriched_or = df_zero_filtered[(df_zero_filtered['G11-18 TPM']> 10 * df_zero_filtered['A11-18 TPM']) | (df_zero_filtered['G12-15 TPM'] > 10 * df_zero_filtered['A12-15 TPM'])]
df_rih_enriched_or

,A11-18 TPM,A12-15 TPM,G11-18 TPM,G12-15 TPM
Name,,,,
nlp-71,121.000891,270.418360,14762.621939,24388.648962
rpl-22,913.453589,1290.676273,13059.914820,18133.134176
lec-8,1016.741583,583.893086,8929.646567,5873.498662
lec-9,1075.730485,804.060270,6867.432783,11608.178426
pdf-1,294.849488,974.611250,4673.600806,2879.218310
...,...,...,...,...
C49C8.5,38.872267,43.669786,80.259857,495.622356
pho-11,45.306905,232.981145,79.500956,3386.358454
vrp-1,16.966073,10.402470,73.497808,135.901480


In [15]:
#Get column names (usually not exactly the same as in the excel)
features = df.columns.tolist()
features

['A11-18 TPM', 'A12-15 TPM', 'G11-18 TPM', 'G12-15 TPM']

#### Data Filtering (Remove Genes)

In [8]:
#Filter out genes if transcripts per million (TPM) < 10 for both L450 and dauer and save to a new Excel file
#If TPM for either L450 or dauer is less than 10, the gene is not filtered, only if both are less than 10
df_filtered_tpm = df[(df['L450 TPM'] >= 10) | (df['D819 TPM'] >= 10)]
df_filtered_tpm.to_excel('Data/D819 vs L450/D819_IP vs. L450_IP_TRAP_simplified filtered by TPM.xlsx', index=False)

#Filter out genes if TPM = 0 for either L450 or dauer
df_zero_filtered_tpm = df[(df['L450 TPM'] > 0) & (df['D819 TPM'] > 0) & (df['L300 TPM'] > 0) & (df['D150 TPM'] > 0)]

In [9]:
#Filter out genes if transcripts per million (RPKM) < 10 for both L450 and dauer and save to a new Excel file
#If RPKM for either L450 or dauer is less than 10, the gene is not filtered, only if both are less than 10
df_filtered_rpkm = df[(df['L450 RPKM'] >= 10) | (df['D819 RPKM'] >= 10)]
df_filtered_rpkm.to_excel('Data/D819 vs L450/D819_IP vs. L450_IP_TRAP_simplified filtered by RPKM.xlsx', index=False)

#Filter out genes if RPKM = 0 for either L450 or dauer
df_zero_filtered_rpkm = df[(df['L450 RPKM'] > 0) & (df['D819 RPKM'] > 0)]

In [10]:
#See filtered results
print("TPM FILTERED RESULTS -----------------------------")
print(f'Before: {df.shape[0]} genes -> After: {df_filtered_tpm.shape[0]} genes')
print(df_filtered_tpm.info())
print(df_filtered_tpm.describe())

print("\nRPKM FILTERED RESULTS -----------------------------")
print(f'Before: {df.shape[0]} genes -> After: {df_filtered_rpkm.shape[0]} genes')
print(df_filtered_rpkm.info())
print(df_filtered_rpkm.describe())

TPM FILTERED RESULTS -----------------------------
Before: 31555 genes -> After: 9542 genes
<class 'pandas.core.frame.DataFrame'>
Index: 9542 entries, 1 to 31554
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Name       9542 non-null   object 
 1   L450 TPM   9542 non-null   float64
 2   L450 RPKM  9542 non-null   float64
 3   D819 TPM   9542 non-null   float64
 4   D819 RPKM  9542 non-null   float64
 5   L300 TPM   9542 non-null   float64
 6   L300 RPKM  9542 non-null   float64
 7   D150 TPM   9542 non-null   float64
 8   L150 RPKM  9542 non-null   float64
dtypes: float64(8), object(1)
memory usage: 745.5+ KB
None
           L450 TPM     L450 RPKM      D819 TPM     D819 RPKM     L300 TPM  \
count   9542.000000   9542.000000   9542.000000   9542.000000  9542.000000   
mean     102.402669     85.229118    102.858524     88.146925    98.979362   
std      432.130993    359.659995    742.225192    636.066566   340.53300

#### Neuropeptide and Related Genes

In [11]:
#From Wormbase.org @ https://www.ncbi.nlm.nih.gov/books/NBK154158/table/neuronalgenome_table15/?report=objectonly
#Also added some additional enzymes and TGF-beta genes from Mei at the end of the list
neuropeptide_genes = [
    "flp-1", "flp-2", "flp-3", "flp-4", "flp-5", "flp-6", "flp-7", "flp-8", "flp-9", "flp-10", "flp-11", "flp-12", "flp-13", "flp-14", "flp-15", "flp-16", 
    "flp-17", "flp-18", "flp-19", "flp-20", "flp-21", "flp-22", "flp-23", "flp-24", "flp-25", "flp-26", "flp-27", "flp-28", "flp-32", "flp-33",  
    "flp-44", "nlp-1", "nlp-2", "nlp-3", "nlp-4", "nlp-5", "nlp-6", "nlp-7", "nlp-8", "nlp-9", "nlp-10", "nlp-11", "nlp-12", "nlp-13", "nlp-14", "nlp-15",
    "nlp-16", "nlp-17", "nlp-18", "nlp-19", "nlp-20", "nlp-21", "nlp-22", "nlp-23", "nlp-24", "nlp-25", "nlp-26", "nlp-27", "nlp-28", "nlp-29", "nlp-30",
    "nlp-31", "nlp-32", "nlp-33", "nlp-34", "nlp-35", "nlp-36", "nlp-37", "nlp-38", "nlp-39", "nlp-40", "nlp-41", "nlp-42", "nlp-43", "nlp-44", "nlp-45",
    "nlp-46", "nlp-47", "nlp-48", "daf-28", "ins-1", "ins-2", "ins-3", "ins-4", "ins-5", "ins-6", "ins-7", "ins-8", "ins-9", "ins-10", "ins-11", "ins-12", 
    "ins-13", "ins-14", "ins-15", "ins-16", "ins-17", "ins-18", "ins-19", "ins-20", "ins-21", "ins-22", "ins-23", "ins-24", "ins-25", "ins-26", "ins-27",
    "ins-28", "ins-29", "ins-30", "ins-31", "ins-32", "ins-33", "ins-34", "ins-35", "ins-36", "ins-37", "ins-38", "ins-39", "pdf-1", "snet-1", "ntc-1", 
    "daf-7", "egl-3", "elg-21", "sbt-1"
]

print(f'Total number of ion channel genes: {len(neuropeptide_genes)}')
print(f'Total number of unique ion channel genes: {len(set(neuropeptide_genes))}')

Total number of ion channel genes: 126
Total number of unique ion channel genes: 126


In [12]:
#Get dataframes filtered by neuropeptide genes
df_neuropep = df[df['Name'].isin(neuropeptide_genes)]
df_zero_filtered_neuropep = df_zero_filtered_tpm[df_zero_filtered_tpm['Name'].isin(neuropeptide_genes)]
df_filtered_tpm_neuropep = df_filtered_tpm[df_filtered_tpm['Name'].isin(neuropeptide_genes)]
df_filtered_rpkm_neuropep = df_filtered_rpkm[df_filtered_rpkm['Name'].isin(neuropeptide_genes)]

print("NEUROPEPTIDE UNFILTERED RESULTS -----------------------------")
print(f'Number of neuropeptide and related genes found in unfiltered dataset: {df_neuropep.shape[0]}\n')
print("NEUROPEPTIDE ZERO FILTERED RESULTS -----------------------------")
print(f'Number of neuropeptide and related genes found in zero filtered dataset: {df_zero_filtered_neuropep.shape[0]}\n')
print("TPM NEUROPEPTIDE FILTERED RESULTS -----------------------------")
print(f'Number of neuropeptide and related genes found in filtered dataset: {df_filtered_tpm_neuropep.shape[0]}\n')
print("RPKM NEUROPEPTIDE FILTERED RESULTS -----------------------------")
print(f'Number of neuropeptide and related genes found in filtered dataset: {df_filtered_rpkm_neuropep.shape[0]}')

NEUROPEPTIDE UNFILTERED RESULTS -----------------------------
Number of neuropeptide and related genes found in unfiltered dataset: 118

NEUROPEPTIDE ZERO FILTERED RESULTS -----------------------------
Number of neuropeptide and related genes found in zero filtered dataset: 114

TPM NEUROPEPTIDE FILTERED RESULTS -----------------------------
Number of neuropeptide and related genes found in filtered dataset: 99

RPKM NEUROPEPTIDE FILTERED RESULTS -----------------------------
Number of neuropeptide and related genes found in filtered dataset: 97


#### Ion Channel Genes

In [13]:
# C.elegans Ion Channel Genes by Subcategory (from NCBI Bookshelf)
# Source: https://www.ncbi.nlm.nih.gov/books/NBK154158/#neuronalgenome_sec2

# 1. Potassium Channels (Table 2; 72 genes)
potassium_channels = [
    # 6-transmembrane domain K+ channels
    "shk-1",  # Shaker/Kv1
    "exp-2", "kvs-1", "kvs-2", "kvs-3", "kvs-4", "kvs-5",  # Shab/Kv2
    "shw-1", "egl-36", "shw-3",  # Shaw/Kv3
    "shl-1",  # Shal/Kv4
    "kqt-1", "kqt-2", "kqt-3",  # KQT family
    "egl-2", "unc-103",  # Eag-like (Kv10–12)
    "slo-1", "slo-2",  # Slo (BK-type)
    "kcnl-1", "kcnl-2", "kcnl-3", "kcnl-4",  # SK-type

    # 4-transmembrane domain TWK (K2P-like) channels
    "egl-23", "sup-9", "unc-58", "unc-110", "twk-1", "twk-2", "twk-3", "twk-4",
    "twk-5", "twk-6", "twk-7", "twk-8", "twk-9", "twk-10", "twk-11", "twk-12",
    "twk-13", "twk-14", "twk-16", "twk-17", "twk-20", "twk-21", "twk-22", "twk-23",
    "twk-24", "twk-25", "twk-26", "twk-28", "twk-29", "twk-30", "twk-31", "twk-32",
    "twk-33", "twk-34", "twk-35", "twk-36", "twk-37", "twk-39", "twk-40", "twk-42",
    "twk-43", "twk-44", "twk-45", "twk-46", "twk-47", "twk-48", "twk-49",

    # 2-transmembrane domain Kir channels
    "irk-1", "irk-2", "irk-3"
]

# 2. Candidate Auxiliary Subunits for Ion Channels (Table 3; 96 genes, contains duplicates)
auxiliary_subunits = [
    # Voltage-gated K+ channel auxiliary subunits
    "mps-1", "mps-2", "mps-3", "mps-4", "K01A2.9", "K01A2.12", "K01A2.3", "K01A2.4", 
    "C25A8.2", "R02D5.7", "F30A10.12", "F53G12.8", "sssh-1", "ncs-6", "ncs-7", "ncs-8", "ncs-9",
    "dpf-1", "dpf-2", "dpf-3", "dpf-4", "dpf-5", "dpf-6", "dpf-7",
    "mec-14", "ctf-1", "mrp-1", "mrp-2", "mrp-3", "mrp-4", "mrp-7", "mrp-8", "mrp-5", "mrp-6",
    "bkip-1",

    # TWK-type potassium channel auxiliary subunits
    "sup-10", "unc-93", "Y39B6A.27", "Y39B6A.29", "Y37A1A.2", "ZK6.6", "ZK6.8", "B0554.5", 
    "B0554.7", "C08D8.1", "C27C12.4", "F31D5.2", "F31D5.1", "M153.2", "Y11D7A.3", "Y39D8A.1", 
    "Y52E8A.4", "F36G9.3",

    # Voltage-gated calcium channel auxiliary subunits
    "unc-79", "unc-80",

    # nAChR LGIC auxiliary subunits
    "lev-9", "T07H6.4", "lev-10", "mig-13", "neto-1", "K05C4.11", "molo-1", "R02D5.3",
    "F15B9.10", "F01D5.6", "Y54E2A.10", "Y12A6A.1", "C09B8.3",
    "lurp-1", "lurp-2", "lurp-3", "lurp-4", "odr-2", "hot-1", "hot-2", "hot-3", "hot-4", "hot-5",
    "hot-6", "hot-7", "hot-8", "hot-9",

    # AMPA-type Glu receptor auxiliary subunits
    "sol-1", "stg-1", "stg-2", "cni-1",

    # DEG/ENaC and TRP-like channel auxiliary subunits
    "mec-2", "unc-1", "unc-24", "stl-1", "sto-1", "sto-2", "sto-3", "sto-4", "sto-5", "sto-6"
]

# 3. Voltage-Gated Calcium Channels (Table 4; 9 genes)
calcium_channels = [
    # α1 subunits
    'egl-19', 'cca-1', 'unc-2', 'nca-1', 'nca-2',
    # β subunits
    'ccb-1', 'ccb-2',
    # α2δ subunits
    'unc-36', 'tag-180'
]

# 4. SLC Transporters with Neuronal Functions (Table 5; 82 genes)
slc_transporters = [
    # SLC17: Vesicular glutamate transporter family (14 genes)
    "eat-4", "vglu-2", "vglu-3", "ZK54.1", "C38C10.2", "vnut-1", "C02C2.4", "T28F3.4", "F21F8.11",
    "F12B6.2", "ZK682.2", "F25G6.7", "T09B9.2", "F45E4.11",

    # SLC18: Vesicular amine transporter family (2 genes)
    "cat-1", "unc-17",

    # SLC32: Vesicular inhibitory amino acid transporter family (1 gene)
    "unc-47",

    # SLC1: High-affinity glutamate and neutral amino acid transporter family (6 genes)
    "glt-1", "glt-3", "glt-4", "glt-5", "glt-6", "glt-7",

    # SLC6: Na+/Cl- dependent neurotransmitter transporter family (17 genes)
    "dat-1", "mod-5", "snf-1", "snf-2", "snf-3", "snf-4", "snf-5", "snf-6", "snf-7", "snf-8",
    "snf-9", "snf-10", "snf-11", "snf-12", "F56F4.3", "C09E8.1", "Y43D4A.1",

    # SLC28: Na+ coupled nucleoside transporter family (2 genes)
    "F27E11.1", "F27E11.2",

    # SLC29: Facilitative nucleoside transporters (7 genes)
    "ent-1", "ent-2", "ent-3", "ent-4", "ent-5", "ent-6", "ent-7",

    # SLC8 & SLC24: Na+/Ca2+ exchanger & Na+/Ca2+-K+ exchanger (10 genes)
    "ncx-1", "ncx-2", "ncx-3", "ncx-4", "ncx-5", "ncx-6", "ncx-7", "ncx-8", "ncx-9", "ncx-10",

    # SLC30: cation diffusion facilitator (CDF) family (12 genes)
    "cdf-2", "ttm-1", "cdf-1", "Y105E8A.3", "toc-1", "Y71H2AM.9", "F41C6.7", "F56C9.3", "K07G5.5",
    "PDB1.1", "ZK185.5", "R02F11.3",

    # SLC12: cation-chloride cotransporter family (7 genes)
    "kcc-1", "kcc-2", "kcc-3", "nkcc-1", "F10E7.9", "B0303.11", "T04B8.5",

    # SLC4: Cl−–HCO3− exchangers (4 genes)
    "abts-1", "abts-2", "abts-3", "abts-4"
]

# 5. Calcium Binding Proteins – “EF hand-only” (Table 6; 65 genes)
calcium_binding_proteins = [
    # Calmodulin family (9 genes)
    "cmd-1", "cal-1", "cal-2", "cal-3", "cal-4", "cal-5", "cal-6", "cal-7", "cal-8",

    # NCS family (7 genes)
    "ncs-1", "ncs-2", "ncs-3", "ncs-4", "ncs-5", "ncs-6", "ncs-7",

    # Others (49 genes)
    "cnb-1", "rsa-1", "C06G1.5", "T22D1.5", "cex-1", "cex-2", "efdh-1", "F59D6.7", "R08D7.5",
    "C56C10.9", "F55A11.1", "T04F3.4", "C29E4.14", "ZK856.8", "pbo-1", "F59D6.7", "micu-1",
    "calm-1", "calu-1", "calu-2", "nucb-1", "T04F8.6", "reps-1", "mlc-1", "mlc-2", "mlc-3",
    "mlc-4", "mlc-5", "mlc-6", "mlc-7", "tnc-1", "tnc-2", "F43C9.2", "cbn-1", "B0563.7",
    "C50C3.5", "C56A3.6", "E02A10.3", "F23F1.2", "H10E21.4", "K03A1.4", "M04F3.4", "R09H10.6",
    "T03F1.11", "T04F3.4", "Y73C8B.5", "F16F9.3", "T09B4.4", "T02G5.2"
]

# 6. TRP Channels (Table 7; 23 genes)
trp_channels = [
    # TRPV family
    "ocr-1", "ocr-2", "ocr-3", "ocr-4", "osm-9",

    # TRPP family
    "lov-1", "pkd-2",

    # TRPN family
    "trp-4",

    # TRPML family
    "cup-5",

    # TRPM family
    "ced-11", "gon-2", "gtl-1", "gtl-2",

    # TRPC family
    "trp-1", "trp-2", "spe-41",  # spe-41 (trp-3) included as "spe-41"

    # TRPA family
    "trpa-1", "trpa-2",

    # TRPM-related2 (TF315286) family
    "trpl-1", "trpl-2", "trpl-3", "trpl-4", "trpl-5"
]

# 7. Cyclic Nucleotide Gated Channels (Table 8; 6 genes)
cng_channels = [
    "tax-2", "tax-4", "cng-1", "cng-2", "cng-3", "che-6"
]

# 8. nAChR-type Ligand-Gated Ion Channels (Cys-loop LGIC, Table 9; 61 genes)
nachr_channels = [
    # ACh receptor subunits (acr genes)
    "acr-2", "acr-3", "acr-5", "acr-6", "acr-7", "acr-8", "acr-9", "acr-10", "acr-11",
    "acr-12", "acr-13", "acr-14", "acr-15", "acr-16", "acr-17", "acr-18", "acr-19", "acr-20",
    "acr-21", "acr-23", "acr-24", "acr-25", "cup-4", "deg-3", "des-2", "eat-2", "lev-1",
    "unc-29", "unc-38", "unc-63",

    # Proton-gated/ligand-gated chloride channels (lgc genes)
    "lgc-1", "lgc-2", "lgc-3", "lgc-4", "lgc-5", "lgc-6", "lgc-7", "lgc-8", "lgc-9", "lgc-10",
    "lgc-11", "lgc-12", "lgc-13", "lgc-14", "lgc-15", "lgc-16", "lgc-17", "lgc-18", "lgc-19",
    "lgc-20", "lgc-21", "lgc-22", "lgc-23", "lgc-24", "lgc-25", "lgc-26", "lgc-27", "lgc-28",
    "lgc-29", "lgc-30", "lgc-31"
]


# 9. Other Ligand-Gated Ion Channels (Cys-loop LGIC, Table 10; 41 genes)
other_ligand_gated_channels = [
    # GABA subgroup
    "gab-1", "unc-49", "exp-1", "lgc-35", "lgc-36", "lgc-37", "lgc-38",

    # Aminergic subgroup
    "mod-1", "lgc-50", "lgc-51", "lgc-52", "lgc-53", "lgc-54", "lgc-55", "ggr-3",

    # GluCl subgroup (Glutamate-gated chloride channels)
    "avr-14", "avr-15", "glc-1", "glc-2", "glc-3", "glc-4",

    # ACC subgroup (Acetylcholine-gated chloride channels)
    "acc-1", "acc-2", "acc-3", "acc-4", "lgc-46", "lgc-47", "lgc-48", "lgc-49",

    # Diverse group
    "lgc-32", "lgc-33", "lgc-34", "ggr-1", "ggr-2", "lgc-39", "lgc-40", "lgc-41",
    "lgc-42", "lgc-43", "lgc-44", "lgc-45"
]


# 10. Ionotropic Glutamate Receptors (Table 11; 15 genes)
glutamate_receptors = [
    # NMDA-type glutamate receptors
    "nmr-1", "nmr-2",

    # AMPA-type glutamate receptors
    "glr-1", "glr-2", "glr-3", "glr-4", "glr-5", "glr-6", "glr-7", "glr-8",

    # Diverse2 group
    "ZK867.2", "C08B6.5", "W02A2.5", "F59E12.8", "T25E4.2"
]


# 11. DEG/ENaC Channels (Table 12; 32 genes)
deg_enac_channels = [
    # Subgroup 1 DEG/ENaC family
    "flr-1", "acd-1", "acd-2", "acd-3", "acd-4", "acd-5",
    "delm-1", "delm-2",

    # Subgroup 2 DEG/ENaC family (ASIC-like)
    "asic-1", "asic-2", "mec-4", "del-4", "unc-105", "deg-1", "del-1", "mec-10",

    # Subgroup 3 (EGF + ASC domain)
    "egas-1", "egas-2", "egas-3", "egas-4",

    # Other DEG/ENaC family members
    "degt-1", "del-2", "del-3", "del-5", "del-6", "del-7", "del-8", "unc-8", "del-9", "del-10",

    # Pseudogenes or fragments
    "Y57G11C.44", "F58G6.8"
]


# 12. Chloride Channels (Table 13; 35 genes)
chloride_channels = [
    # CLC-type chloride channels (6 genes)
    "clh-1", "clh-2", "clh-3", "clh-4", "clh-5", "clh-6",

    # Anoctamin-related chloride channels (2 genes)
    "anoh-1", "anoh-2",

    # Tweety-related chloride channel (1 gene)
    "ttyh-1",

    # Bestrophin-related chloride channels (26 genes)
    "best-1", "best-2", "best-3", "best-4", "best-5", "best-6", "best-7", "best-8", "best-9",
    "best-10", "best-11", "best-12", "best-13", "best-14", "best-15", "best-16", "best-17",
    "best-18", "best-19", "best-20", "best-21", "best-22", "best-23", "best-24", "best-25", "best-26"
]

ion_channel_genes = (
    potassium_channels +
    auxiliary_subunits +
    calcium_channels +
    slc_transporters +
    calcium_binding_proteins +
    trp_channels +
    cng_channels +
    nachr_channels +
    other_ligand_gated_channels +
    glutamate_receptors +
    deg_enac_channels +
    chloride_channels
)
# Print the total number of ion channel genes
print(f'Total number of ion channel genes: {len(ion_channel_genes)}')
print(f'Total number of unique ion channel genes: {len(set(ion_channel_genes))}')


Total number of ion channel genes: 537
Total number of unique ion channel genes: 533


In [14]:
#Get dataframes filtered by ion channel genes
df_ion_channels = df[df['Name'].isin(ion_channel_genes)]
df_zero_filtered_ion_channels = df_zero_filtered_tpm[df_zero_filtered_tpm['Name'].isin(ion_channel_genes)]
df_filtered_tpm_ion_channels = df_filtered_tpm[df_filtered_tpm['Name'].isin(ion_channel_genes)]
df_filtered_rpkm_ion_channels = df_filtered_rpkm[df_filtered_rpkm['Name'].isin(ion_channel_genes)]

print("ION CHANNEL UNFILTERED RESULTS -----------------------------")
print(f'Number of ion channel genes found in unfiltered dataset: {df_ion_channels.shape[0]}\n')
print("ION CHANNEL ZERO FILTERED RESULTS -----------------------------")
print(f'Number of ion channel genes found in zero filtered dataset: {df_zero_filtered_ion_channels.shape[0]}\n')
print("TPM ION CHANNEL FILTERED RESULTS -----------------------------")
print(f'Number of ion channel genes found in filtered dataset: {df_filtered_tpm_ion_channels.shape[0]}\n')
print("RPKM ION CHANNEL FILTERED RESULTS -----------------------------")
print(f'Number of ion channel genes found in filtered dataset: {df_filtered_rpkm_ion_channels.shape[0]}')

ION CHANNEL UNFILTERED RESULTS -----------------------------
Number of ion channel genes found in unfiltered dataset: 486

ION CHANNEL ZERO FILTERED RESULTS -----------------------------
Number of ion channel genes found in zero filtered dataset: 467

TPM ION CHANNEL FILTERED RESULTS -----------------------------
Number of ion channel genes found in filtered dataset: 293

RPKM ION CHANNEL FILTERED RESULTS -----------------------------
Number of ion channel genes found in filtered dataset: 274


#### GPCR genes

In [15]:
# Source: https://www.ncbi.nlm.nih.gov/books/NBK154158/#neuronalgenome_sec2

# 1. Metabotropic neurotransmitter receptors (Table 19; 27 genes)
metabotropic_receptors = [
    # Metabotropic Glutamate Receptors (5 genes)
    "mgl-1", "mgl-2", "mgl-3", "C30A5.10", "F35H10.10",

    # Muscarinic Acetylcholine Receptors (mAChRs) (3 genes)
    "gar-1", "gar-2", "gar-3",

    # Metabotropic GABA Receptors (2 genes)
    "gbb-1", "gbb-2",

    # Biogenic Amine Receptors (16 genes)
    "dop-1", "dop-2", "dop-3", "dop-4", "dop-5", "dop-6",
    "octr-1",
    "ser-3", "ser-6", "ser-1", "ser-4", "ser-5", "ser-7",
    "ser-2",
    "tyra-2", "tyra-3",

    # Adenosine Receptor (1 gene)
    "ador-1"
]

# 2. GPCR putative neuropeptide receptors (Table 20; 102 genes)
gpcr_receptors = [
    # Class B (secretin-type)
    "pdfr-1", "seb-2", "seb-3",
    
    # Neuropeptide F/Y receptor family
    "npr-1", "npr-2", "npr-3", "npr-4", "npr-5", "npr-6", "npr-7", "npr-8", 
    "npr-10", "npr-11", "npr-12", "npr-13",
    
    # Ghrelin-obstatin/neuromedin U receptor family
    "nmur-1", "nmur-2", "nmur-3", "nmur-4", "npr-20", "npr-21",
    
    # Neurokinin/neuropeptide FF/orexin receptor family
    "tkr-1", "tkr-2", "tkr-3", "npr-14", "npr-22", "npr-35",
    
    # Somatostatin receptor family
    "npr-16", "npr-17", "npr-18", "npr-24", "npr-25", "npr-26", "npr-27", 
    "npr-28", "npr-29", "npr-30", "npr-31", "npr-32",
    
    # Galanin receptor family
    "npr-9", "npr-15", "npr-33", "npr-34",
    
    # Gonadotropin-releasing hormone receptor family
    "gnrr-1", "gnrr-2", "gnrr-3", "gnrr-4", "gnrr-5", "gnrr-6", "gnrr-7", "daf-38",
    
    # Gastrin-cholecystokinin receptor family
    "ckr-1", "ckr-2",
    
    # Vasopressin receptor family
    "ntr-1", "ntr-2",
    
    # Sex peptide receptor family
    "sprr-1", "sprr-2", "sprr-3",
    
    # Drosophila FMRFamide receptor family
    "frpr-1", "frpr-2", "frpr-3", "frpr-4", "frpr-5", "frpr-6", "frpr-7", "frpr-8",
    "frpr-9", "frpr-10", "frpr-11", "frpr-12", "frpr-13", "frpr-14", "frpr-15",
    "frpr-16", "frpr-17", "frpr-18", "frpr-19", "daf-37",
    
    # Dromyosuppressin receptor family
    "egl-6", "dmsr-1", "dmsr-2", "dmsr-3", "dmsr-4", "dmsr-5", "dmsr-6", "dmsr-7",
    "dmsr-8", "dmsr-9", "dmsr-10", "dmsr-11", "dmsr-12", "dmsr-13", "dmsr-14", 
    "dmsr-15", "dmsr-16",
    
    # Miscellaneous receptors
    "aex-2", "aexr-1", "aexr-2", "aexr-3",
    "fshr-1",
    
    # No obvious paralogs/orthologs
    "npr-19", "npr-23"
]

# 3. Adhesion type GPCRs (Table 22; 5 genes)
adhesion_gpcrs = [
    "fmi-1", "lat-1", "lat-2", "mth-1", "mth-2"
]

#Total GPCR genes
gpcr_genes = metabotropic_receptors + gpcr_receptors + adhesion_gpcrs

# Print the total number of GPCR genes
print(f'Total number of GPCR genes: {len(gpcr_genes)}')
print(f'Total number of unique GPCR genes: {len(set(gpcr_genes))}')


Total number of GPCR genes: 134
Total number of unique GPCR genes: 134


In [16]:
#Get dataframes filtered by GPCR genes
df_gpcrs = df[df['Name'].isin(gpcr_genes)]
df_zero_filtered_gpcrs = df_zero_filtered_tpm[df_zero_filtered_tpm['Name'].isin(gpcr_genes)]
df_filtered_tpm_gpcrs = df_filtered_tpm[df_filtered_tpm['Name'].isin(gpcr_genes)]
df_filtered_rpkm_gpcrs = df_filtered_rpkm[df_filtered_rpkm['Name'].isin(gpcr_genes)]

print("GPCR UNFILTERED RESULTS -----------------------------")
print(f'Number of GPCR genes found in unfiltered dataset: {df_gpcrs.shape[0]}\n')
print("GPCR ZERO FILTERED RESULTS -----------------------------")
print(f'Number of GPCR genes found in zero filtered dataset: {df_zero_filtered_gpcrs.shape[0]}\n')
print("TPM GPCR FILTERED RESULTS -----------------------------")
print(f'Number of GPCR genes found in filtered dataset: {df_filtered_tpm_gpcrs.shape[0]}\n')
print("RPKM GPCR FILTERED RESULTS -----------------------------")
print(f'Number of GPCR genes found in filtered dataset: {df_filtered_rpkm_gpcrs.shape[0]}')

GPCR UNFILTERED RESULTS -----------------------------
Number of GPCR genes found in unfiltered dataset: 132

GPCR ZERO FILTERED RESULTS -----------------------------
Number of GPCR genes found in zero filtered dataset: 130

TPM GPCR FILTERED RESULTS -----------------------------
Number of GPCR genes found in filtered dataset: 106

RPKM GPCR FILTERED RESULTS -----------------------------
Number of GPCR genes found in filtered dataset: 106


#### Gap Junction Genes

In [17]:
# Source: https://www.ncbi.nlm.nih.gov/books/NBK154158/#neuronalgenome_sec2

# 1. Metabotropic neurotransmitter receptors (Table 28; 25 genes)
gap_junction_genes = [
    # Neuronally expressed innexins
    "inx-1", "inx-2", "inx-3", "inx-4", "inx-5",
    "inx-6", "inx-7", "inx-8", "inx-9", "inx-10",
    "inx-11", "inx-12", "inx-13", "inx-14", "inx-17",
    "inx-18", "inx-19", "unc-7", "unc-9",

    # Other innexins
    "inx-15", "inx-16", "inx-20", "inx-21", "inx-22", "eat-5"
]

# Print the total number of gap junction genes
print(f'Total number of gap junction genes: {len(gap_junction_genes)}')
print(f'Total number of unique gap junction genes: {len(set(gap_junction_genes))}')


Total number of gap junction genes: 25
Total number of unique gap junction genes: 25


In [18]:
#Get dataframes filtered by gap junction genes
df_gap_junctions = df[df['Name'].isin(gap_junction_genes)]
df_zero_filtered_gap_junctions = df_zero_filtered_tpm[df_zero_filtered_tpm['Name'].isin(gap_junction_genes)]
df_filtered_tpm_gap_junctions = df_filtered_tpm[df_filtered_tpm['Name'].isin(gap_junction_genes)]
df_filtered_rpkm_gap_junctions = df_filtered_rpkm[df_filtered_rpkm['Name'].isin(gap_junction_genes)]

print("GAP JUNCTION UNFILTERED RESULTS -----------------------------")
print(f'Number of gap junction genes found in unfiltered dataset: {df_gap_junctions.shape[0]}\n')
print("GAP JUNCTION ZERO FILTERED RESULTS -----------------------------")
print(f'Number of gap junction genes found in zero filtered dataset: {df_zero_filtered_gap_junctions.shape[0]}\n')
print("TPM GAP JUNCTION FILTERED RESULTS -----------------------------")
print(f'Number of gap junction genes found in filtered dataset: {df_filtered_tpm_gap_junctions.shape[0]}\n')
print("RPKM GAP JUNCTION FILTERED RESULTS -----------------------------")
print(f'Number of gap junction genes found in filtered dataset: {df_filtered_rpkm_gap_junctions.shape[0]}')

GAP JUNCTION UNFILTERED RESULTS -----------------------------
Number of gap junction genes found in unfiltered dataset: 24

GAP JUNCTION ZERO FILTERED RESULTS -----------------------------
Number of gap junction genes found in zero filtered dataset: 23

TPM GAP JUNCTION FILTERED RESULTS -----------------------------
Number of gap junction genes found in filtered dataset: 11

RPKM GAP JUNCTION FILTERED RESULTS -----------------------------
Number of gap junction genes found in filtered dataset: 11


#### Transcription Factor Genes

In [19]:
# Source: https://genomebiology.biomedcentral.com/articles/10.1186/gb-2005-6-13-r110#MOESM1

#Get columns of interest from transcription factor dataframe
df2=df2[['Sequence name', 'Public name']]

# Print the total number of transcription factor genes
print(f'Total number of transcription factor genes: {df2.shape[0]}')


Total number of transcription factor genes: 953


In [20]:
#Get dataframes filtered by transcription factor genes
df_tfs = df[df['Name'].isin(df2['Public name']) | df['Name'].isin(df2['Sequence name'])]
df_zero_filtered_tfs = df_zero_filtered_tpm[df_zero_filtered_tpm['Name'].isin(df2['Public name']) | df_zero_filtered_tpm['Name'].isin(df2['Sequence name'])]
df_filtered_tpm_tfs = df_filtered_tpm[df_filtered_tpm['Name'].isin(df2['Public name']) | df_filtered_tpm['Name'].isin(df2['Sequence name'])]
df_filtered_rpkm_tfs = df_filtered_rpkm[df_filtered_rpkm['Name'].isin(df2['Public name']) | df_filtered_rpkm['Name'].isin(df2['Sequence name'])]

print("TRANSCRIPTION FACTOR UNFILTERED RESULTS -----------------------------")
print(f'Number of transcription factor genes found in unfiltered dataset: {df_tfs.shape[0]}\n')
print("TRANSCRIPTION FACTOR ZERO FILTERED RESULTS -----------------------------")
print(f'Number of transcription factor genes found in zero filtered dataset: {df_zero_filtered_tfs.shape[0]}\n')
print("TPM TRANSCRIPTION FACTOR FILTERED RESULTS -----------------------------")
print(f'Number of transcription factor genes found in filtered dataset: {df_filtered_tpm_tfs.shape[0]}\n')
print("RPKM TRANSCRIPTION FACTOR FILTERED RESULTS -----------------------------")
print(f'Number of transcription factor genes found in filtered dataset: {df_filtered_rpkm_tfs.shape[0]}')

TRANSCRIPTION FACTOR UNFILTERED RESULTS -----------------------------
Number of transcription factor genes found in unfiltered dataset: 641

TRANSCRIPTION FACTOR ZERO FILTERED RESULTS -----------------------------
Number of transcription factor genes found in zero filtered dataset: 568

TPM TRANSCRIPTION FACTOR FILTERED RESULTS -----------------------------
Number of transcription factor genes found in filtered dataset: 338

RPKM TRANSCRIPTION FACTOR FILTERED RESULTS -----------------------------
Number of transcription factor genes found in filtered dataset: 315
